# Chapter 7 Tutorial — Potentials, Excessive Functions, and Optimal Stopping of Markov Chains

This notebook is a step-by-step tutorial for Chapter 7 of Çınlar, based on the uploaded chapter excerpt. It covers:

1. discounted potentials of Markov chains;
2. the matrix formula \(R^\alpha=(I-\alpha P)^{-1}\);
3. stopping-time decompositions of potentials;
4. hitting-time equations;
5. excessive functions and their decomposition;
6. optional sampling for excessive functions;
7. optimal stopping, value functions, and the minimal excessive majorant;
8. computational examples with finite Markov chains.

The main mental model:

> A Markov chain generates a stream of future rewards. A **potential** is the expected discounted total reward. An **excessive function** is a value function that is at least as valuable now as its expected discounted future value. In optimal stopping, the value function is the smallest excessive function above the immediate payoff.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)


def as_array(x):
    return np.array(x, dtype=float)


def potential_matrix(P, alpha=1.0):
    """Return R^alpha = (I - alpha P)^(-1), when invertible."""
    P = as_array(P)
    I = np.eye(P.shape[0])
    return np.linalg.inv(I - alpha * P)


def discounted_potential(P, g, alpha=1.0):
    """Return R^alpha g."""
    return potential_matrix(P, alpha) @ as_array(g)


def simulate_chain(P, start, n, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    P = as_array(P)
    x = start
    path = [x]
    for _ in range(n):
        x = rng.choice(len(P), p=P[x])
        path.append(x)
    return np.array(path)


def simulate_discounted_reward(P, g, start, alpha, horizon=1000, reps=20000, seed=0):
    rng = np.random.default_rng(seed)
    g = as_array(g)
    vals = []
    discounts = alpha ** np.arange(horizon + 1)
    for _ in range(reps):
        path = simulate_chain(P, start, horizon, rng)
        vals.append(np.sum(discounts * g[path]))
    return np.mean(vals), np.std(vals) / np.sqrt(reps)


def first_hit(path, A):
    A = set(A)
    for n, x in enumerate(path):
        if x in A:
            return n
    return None

## 1. Potentials

Let \(X=(X_n:n\in\mathbb N)\) be a Markov chain on state space \(E\) with transition matrix \(P\). Let \(g:E\to\mathbb R_+\) be a reward function, and let \(\alpha\in[0,1]\) be a discount factor.

Starting from state \(i\), the **\(\alpha\)-potential** of \(g\) is

\[
R^\alpha g(i)=E_i\left[\sum_{n=0}^{\infty}\alpha^n g(X_n)\right].
\]

When \(\alpha=1\), this is simply called the **potential** \(Rg\):

\[
Rg(i)=E_i\left[\sum_{n=0}^{\infty} g(X_n)\right].
\]

Thinking model:

- \(g(X_n)\) is the reward at time \(n\).
- \(\alpha^n\) discounts rewards far in the future.
- \(R^\alpha g(i)\) is the expected present value of all future rewards when starting at \(i\).

### Example 1.3: counting visits as a potential

Let \(g(j)=1\) for a fixed state \(j\), and \(g(k)=0\) otherwise. Then

\[
Rg(i)=E_i\left[\sum_{n=0}^{\infty}1_{\{X_n=j\}}\right]
=E_i[N_j]=R(i,j).
\]

So the potential matrix from Chapter 5 is a special case of the potential of a reward function: the reward is \(1\) whenever the chain visits \(j\).

In [ ]:
# A small finite chain where state 2 is absorbing.
P = np.array([
    [0.2, 0.6, 0.2],
    [0.1, 0.6, 0.3],
    [0.0, 0.0, 1.0],
])

# Discounted expected number of visits to state 1.
alpha = 0.8
g = np.array([0, 1, 0])
R_alpha = potential_matrix(P, alpha)
R_alpha_g = R_alpha @ g
R_alpha, R_alpha_g

For \(\alpha<1\), the discounted visit count is finite even if the chain can keep returning forever. For \(\alpha=1\), the potential may be infinite if recurrent states are rewarded.

## 1.1 Matrix form of the potential

For \(\alpha\in[0,1]\), define the matrix

\[
R^\alpha = \sum_{n=0}^{\infty}\alpha^n P^n.
\]

Then

\[
R^\alpha g(i)=\sum_j R^\alpha(i,j)g(j).
\]

### Proof idea

Use the Markov property and the definition of matrix powers:

\[
E_i[g(X_n)] = \sum_j P^n(i,j)g(j).
\]

Then

\[
E_i\left[\sum_{n=0}^{\infty}\alpha^n g(X_n)\right]
=\sum_{n=0}^{\infty}\alpha^n\sum_jP^n(i,j)g(j)
=\sum_j\left(\sum_{n=0}^{\infty}\alpha^nP^n(i,j)\right)g(j).
\]

So the potential is matrix multiplication: \(R^\alpha g\).

### Resolvent equation

The matrix \(R^\alpha\) satisfies

\[
R^\alpha = I + \alpha P R^\alpha = I + \alpha R^\alpha P.
\]

When the inverse exists,

\[
R^\alpha = (I-\alpha P)^{-1}.
\]

Equivalently, \(f=R^\alpha g\) is the unique bounded solution of

\[
f = g+\alpha P f,
\]

or

\[
(I-\alpha P)f=g.
\]

This is the Bellman-style equation for a fixed reward stream.

In [ ]:
# Check the identity R = I + alpha P R numerically.
left = R_alpha
right = np.eye(P.shape[0]) + alpha * P @ R_alpha
np.max(np.abs(left - right))

## 1.2 Proposition 1.5: uniqueness of the bounded solution

Suppose \(0\le \alpha<1\) and \(g\) is bounded and non-negative. The function \(R^\alpha g\) is the unique bounded non-negative solution of

\[
f = g + \alpha P f.
\]

### Proof

First, direct expansion gives

\[
R^\alpha g = g+\alpha P g+\alpha^2P^2g+\cdots.
\]

Therefore

\[
\alpha P R^\alpha g=\alpha Pg+\alpha^2P^2g+\cdots,
\]

so

\[
R^\alpha g = g+\alpha P R^\alpha g.
\]

Now suppose \(f\) is another bounded non-negative solution:

\[
f=g+\alpha Pf.
\]

Substitute the equation into itself repeatedly:

\[
f=g+\alpha Pg+\cdots+\alpha^nP^ng+\alpha^{n+1}P^{n+1}f.
\]

Because \(f\) is bounded and \(\alpha<1\), the final term vanishes as \(n\to\infty\):

\[
\|\alpha^{n+1}P^{n+1}f\|_\infty\le \alpha^{n+1}\|f\|_\infty\to0.
\]

Hence

\[
f=R^\alpha g.
\]

## 1.3 Example 1.9: a two-state potential

Let

\[
P=\begin{pmatrix}
1/2 & 1/2\\
1/4 & 3/4
\end{pmatrix},\qquad g=(1,5)^T.
\]

Then

\[
R^\alpha=(I-\alpha P)^{-1}
\]

and \(R^\alpha g\) gives the discounted total expected reward from states 1 and 2.

In [ ]:
P2 = np.array([[1/2, 1/2], [1/4, 3/4]], dtype=float)
g2 = np.array([1, 5], dtype=float)
alpha = 0.9
R2 = potential_matrix(P2, alpha)
f2 = R2 @ g2
R2, f2

In [ ]:
# Monte Carlo verification of Example 1.9
for start in [0, 1]:
    mean, se = simulate_discounted_reward(P2, g2, start=start, alpha=0.9, horizon=300, reps=20000, seed=42+start)
    print(start, "simulation", mean, "+/-", 2*se, "exact", f2[start])

The simulation estimates the same values as \((I-\alpha P)^{-1}g\). The small difference is Monte Carlo noise plus truncation of the infinite sum.

## 1.4 Example 1.10: replacement/maintenance cost

The chapter considers a machine whose remaining useful life is \(j\). If the machine is alive, it ages one year. When it fails, it is replaced by a new machine whose lifetime has distribution

\[
p_i=P(\text{new lifetime}=i)=(0.4)(0.6)^{i-1},\qquad i=1,2,\ldots.
\]

The reward/cost function is

\[
g=(1200,600,240,100,100,100,\ldots).
\]

The interest rate is 20%, so one dollar next year is worth \(\alpha=1/1.2=5/6\) dollars today.

The desired quantity is the discounted maintenance cost starting with a new machine whose lifetime has the geometric distribution above.

The chapter solves the infinite linear system recursively and obtains

\[
f_0=4600.80.
\]

Below we approximate this calculation with a large finite truncation.

In [ ]:
def replacement_chain(N=60, p=0.4):
    """States 0..N represent remaining lifetime. 0 means failed/replaced immediately.
    Truncate the geometric replacement distribution to states 1..N.
    """
    P = np.zeros((N+1, N+1))
    probs = np.array([p*(1-p)**(i-1) for i in range(1, N+1)])
    probs[-1] += 1 - probs.sum()  # tail mass into N
    P[0, 1:] = probs
    for j in range(1, N+1):
        P[j, j-1] = 1.0
    return P, probs

N = 80
alpha = 5/6
P_rep, probs = replacement_chain(N)
g_rep = np.ones(N+1) * 100
g_rep[0] = 1200
g_rep[1] = 600
g_rep[2] = 240

f_rep = discounted_potential(P_rep, g_rep, alpha)
starting_distribution = np.r_[0, probs]  # new machine lifetime starts in 1..N
value_new_machine = starting_distribution @ f_rep
f_rep[:8], value_new_machine

The truncated finite computation is close to the book's infinite-state value. Increasing the truncation makes the approximation stable because the geometric tail decays quickly.

In [ ]:
values = []
for N in [10, 20, 40, 80, 160]:
    P_rep, probs = replacement_chain(N)
    g_rep = np.ones(N+1) * 100
    g_rep[0] = 1200
    g_rep[1] = 600
    g_rep[2] = 240
    f_rep = discounted_potential(P_rep, g_rep, 5/6)
    values.append((N, np.r_[0, probs] @ f_rep))
pd.DataFrame(values, columns=["truncation_N", "discounted_cost"])

## 1.5 Stopping-time decomposition of a potential

Let \(T\) be a stopping time. Then for non-negative \(g\),

\[
R^\alpha g(i)
=E_i\left[\sum_{n=0}^{T-1}\alpha^n g(X_n)\right]
+E_i\left[\alpha^T R^\alpha g(X_T)\right].
\]

Interpretation:

- Accumulate rewards before time \(T\).
- At time \(T\), restart the problem from the random state \(X_T\).
- Discount the restarted value by \(\alpha^T\).

This is a direct consequence of the strong Markov property.

### Proof sketch

Split the infinite sum:

\[
\sum_{n=0}^{\infty}\alpha^ng(X_n)
=\sum_{n=0}^{T-1}\alpha^ng(X_n)
+\alpha^T\sum_{m=0}^{\infty}\alpha^m g(X_{T+m}).
\]

Given the history up to \(T\), the post-\(T\) process behaves like a fresh Markov chain started at \(X_T\). Therefore

\[
E_i\left[\sum_{m=0}^{\infty}\alpha^m g(X_{T+m})\mid \mathcal F_T\right]
=R^\alpha g(X_T).
\]

Taking expectations gives the formula.

## 1.6 Example 1.15: expected discounted cost until second failure

Continuing the replacement example, let \(T\) be the time of the second failure. The chapter computes an expected discounted cost using the stopping-time decomposition and the earlier value function. The key point is not the arithmetic itself, but the method:

\[
a = E_3\left[\sum_{n=0}^{T-1}\alpha^n g(X_n)\right]
=R^\alpha g(3)-E_3[\alpha^T R^\alpha g(X_T)].
\]

At the second failure, \(X_T=0\), so

\[
a=R^\alpha g(3)-R^\alpha g(0)E_3[\alpha^T].
\]

The chapter evaluates this expression as

\[
a=3379.17-1775.00=1604.17.
\]

The thinking model: compute the total infinite value, then subtract the discounted value of the future after the stopping time.

## 1.7 Hitting-time equation

Let \(T\) be the first visit to a set \(A\):

\[
T=\inf\{n\ge0:X_n\in A\}.
\]

For bounded \(g\), define

\[
h(i)=E_i\left[\sum_{n=0}^{T-1}\alpha^n g(X_n)\right].
\]

Then

\[
h(i)=
\begin{cases}
0, & i\in A,\\
g(i)+\alpha\sum_j P(i,j)h(j), & i\notin A.
\end{cases}
\]

This is the standard first-step equation: if you are already in the target set, the pre-hit reward is zero; otherwise collect today's reward and continue from tomorrow's state.

### Example 1.19: solving a hitting reward system

Let \(E=\{a,b,c,d\}\), and

\[
P=\begin{pmatrix}
0.3&0.7&0&0\\
0.4&0.3&0.3&0\\
0&0&0.5&0.5\\
0&0&1&0
\end{pmatrix}.
\]

Compute

\[
h(a)=E_a\left[\sum_{n=0}^{T-1}g(X_n)\right]
\]

for \(g=(5,3,1,4)\) and \(T=\inf\{n:X_n=d\}\).

Because \(d\in A\), \(h(d)=0\). The equations are

\[
\begin{aligned}
h(a)&=5+0.3h(a)+0.7h(b),\\
h(b)&=3+0.4h(a)+0.3h(b)+0.3h(c),\\
h(c)&=1+0.5h(c),\\
h(d)&=0.
\end{aligned}
\]

Solving gives

\[
h(c)=2,
\qquad 0.7h(b)=3.6+0.4h(a),
\qquad h(a)=\frac{86}{9}.
\]

In [ ]:
states = ['a', 'b', 'c', 'd']
P4 = np.array([
    [0.3, 0.7, 0.0, 0.0],
    [0.4, 0.3, 0.3, 0.0],
    [0.0, 0.0, 0.5, 0.5],
    [0.0, 0.0, 1.0, 0.0],
])
g4 = np.array([5, 3, 1, 4], dtype=float)
A = [3]  # d
not_A = [0, 1, 2]

# Solve h_N = g_N + P_NN h_N, h_A = 0
Q = P4[np.ix_(not_A, not_A)]
h_not_A = np.linalg.solve(np.eye(len(not_A)) - Q, g4[not_A])
h = np.zeros(4)
h[not_A] = h_not_A
pd.Series(h, index=states)

## 2. Excessive functions

Let \(f:E\to\mathbb R_+\). For \(\alpha\in[0,1]\), \(f\) is **\(\alpha\)-excessive** if

\[
f\ge \alpha P f.
\]

If \(\alpha=1\), \(f\) is simply called **excessive**.

Interpretation:

\[
f(i)\ge E_i[\alpha f(X_1)].
\]

So \(f(i)\) is at least as large as the expected discounted value after one step. In finance/stopping language, \(f\) is a superharmonic value function: waiting one step cannot increase expected discounted value beyond today's value.

### Basic closure properties

If \(f\) and \(g\) are \(\alpha\)-excessive, then so are:

\[
f\wedge g,\qquad f+g,\qquad cf\quad(c>0).
\]

If \(f\) is \(\alpha\)-excessive and \(\beta<\alpha\), then \(f\) is also \(\beta\)-excessive, because

\[
f\ge \alpha Pf\ge \beta Pf.
\]

## 2.1 Every potential is excessive

If \(f=R^\alpha g\) for non-negative \(g\), then \(f\) is \(\alpha\)-excessive.

### Proof

Since

\[
f=g+\alpha Pf,
\]

and \(g\ge0\), we have

\[
f\ge \alpha Pf.
\]

So potentials are excessive.

In [ ]:
# Check f >= alpha P f for the two-state potential example.
f = f2
alpha = 0.9
alpha_Pf = alpha * P2 @ f
pd.DataFrame({"f": f, "alpha_Pf": alpha_Pf, "gap": f - alpha_Pf})

## 2.2 Riesz-type decomposition

The chapter proves a decomposition theorem:

If \(f\) is \(\alpha\)-excessive, then

\[
f=R^\alpha g+h,
\]

where

\[
g=f-\alpha Pf\ge0,
\]

and \(h\) is \(\alpha\)-harmonic:

\[
h=\alpha Ph.
\]

Thinking model:

- \(g=f-\alpha Pf\) is the **excess mass** or **one-step surplus**.
- \(R^\alpha g\) is the accumulated future surplus.
- \(h\) is the part that persists forever without decay under the transition operator.

For \(\alpha<1\) and bounded \(f\), the harmonic part is zero, so every bounded \(\alpha\)-excessive function is a potential.

### Proof sketch

Start from

\[
f=g+\alpha Pf,
\qquad g=f-\alpha Pf\ge0.
\]

Iterate:

\[
f=g+\alpha Pg+\cdots+\alpha^nP^ng+\alpha^{n+1}P^{n+1}f.
\]

The partial potential term converges to \(R^\alpha g\). The remaining term

\[
h=\lim_{n\to\infty}\alpha^{n+1}P^{n+1}f
\]

satisfies \(h=\alpha Ph\). Therefore \(f=R^\alpha g+h\).

## 2.3 Optional sampling for excessive functions

If \(f\) is \(\alpha\)-excessive, then for any stopping time \(T\),

\[
f(i)\ge E_i[\alpha^T f(X_T)].
\]

This generalizes the one-step inequality \(f\ge \alpha Pf\) to random stopping times.

If \(T\le S\) are stopping times, then

\[
E_i[\alpha^T f(X_T)]\ge E_i[\alpha^S f(X_S)].
\]

Thinking model: for an excessive function, the discounted process \(\alpha^n f(X_n)\) behaves like a supermartingale. Optional sampling says stopping later cannot increase its expected value.

### Irreducible recurrent chains

If \(X\) is irreducible recurrent and \(f\) is excessive, then \(f\) is constant.

Reason: from any state \(i\), the chain eventually hits any other state \(j\) with probability 1. Applying optional sampling to the hitting time of \(j\) gives

\[
f(i)\ge f(j).
\]

Swapping \(i\) and \(j\) gives \(f(j)\ge f(i)\). Hence \(f(i)=f(j)\) for all states.

## 3. Optimal stopping

Let \(f:E\to\mathbb R\) be a bounded payoff function. At each time \(n\), after observing \(X_n, we may stop and receive \(f(X_n)\), or continue.

The value function is

\[
v(i)=\sup_T E_i[f(X_T)],
\]

where the supremum is over all stopping times.

A stopping time \(T_0\) is optimal if

\[
v(i)=E_i[f(X_{T_0})]
\]

for all \(i\).

### Example 3.1: a gambling machine

The machine has states

\[
E=\{A,K,Q,J,2\}
\]

with payoff

\[
f=(6,5,4,3,0).
\]

The player may stop or continue. The transition matrix in the chapter is

\[
P=\begin{pmatrix}
1&0&0&0&0\\
1/3&0&1/3&0&1/3\\
1/4&0&0&1/4&1/2\\
0&3/4&1/4&0&0\\
0&0&0&0&1
\end{pmatrix}.
\]

States \(A\) and \(2\) are absorbing. Starting from \(A\) or \(2\), stopping immediately is clearly optimal. The chapter computes:

- \(v(A)=6\), \(v(2)=0\);
- at \(Q\), continuing gives \(4.5>f(Q)=4\), so \(v(Q)=4.5\);
- at \(K\), continuing gives \(5\), equal to stopping;
- at \(J\), continuing gives \(4.833>3\), so \(v(J)=4.833\).

Therefore

\[
v=(6,5,4.5,4.833\ldots,0).
\]

An optimal rule is: stop when the chain first enters \(\{A,K,2\}\).

In [ ]:
states = ['A', 'K', 'Q', 'J', '2']
P_game = np.array([
    [1,   0,   0,   0,   0],
    [1/3, 0,   1/3, 0,   1/3],
    [1/4, 0,   0,   1/4, 1/2],
    [0,   3/4, 1/4, 0,   0],
    [0,   0,   0,   0,   1],
], dtype=float)
payoff = np.array([6, 5, 4, 3, 0], dtype=float)

def value_iteration_stopping(P, f, iters=1000, tol=1e-12):
    v = f.copy().astype(float)
    history = [v.copy()]
    for _ in range(iters):
        new = np.maximum(f, P @ v)
        history.append(new.copy())
        if np.max(np.abs(new - v)) < tol:
            return new, np.array(history)
        v = new
    return v, np.array(history)

v_game, hist_game = value_iteration_stopping(P_game, payoff)
pd.DataFrame({"payoff f": payoff, "value v": v_game, "continue value Pv": P_game @ v_game}, index=states)

In [ ]:
# Visualize convergence of value iteration.
for i, s in enumerate(states):
    plt.plot(hist_game[:, i], label=s)
plt.xlabel("iteration")
plt.ylabel("estimated value")
plt.title("Value iteration for optimal stopping example")
plt.legend()
plt.show()

## 3.1 The value function as the smallest excessive majorant

The central theorem says:

\[
v\text{ is the minimal excessive function satisfying }v\ge f.
\]

Equivalently, if \(g\) is excessive and \(g\ge f\), then \(g\ge v\).

### Proof idea

First, \(v\ge f\), because stopping immediately is allowed.

Second, \(v\ge Pv\): after one step, we could still follow an optimal strategy from the new state, so continuing one step cannot have value more than \(v\). Therefore \(v\) is excessive.

Third, if \(g\ge f\) and \(g\) is excessive, then optional sampling gives

\[
g(i)
\ge E_i[g(X_T)]
\ge E_i[f(X_T)]
\]

for every stopping time \(T\). Taking the supremum over \(T\) gives \(g(i)\ge v(i)\). Thus \(v\) is the smallest excessive majorant.

### Computational note

For a finite state space, finding \(v\) can be formulated as a linear programming problem:

Minimize

\[
\sum_i v(i)
\]

subject to

\[
v\ge f,
\qquad v\ge Pv,
\qquad v\ge0.
\]

Since \(v\ge Pv\) is the excessivity condition, the optimizer is the smallest excessive majorant.

Value iteration computes the same object by repeatedly applying

\[
v_{n+1}=\max(f,Pv_n).
\]

## 3.2 Example 3.6: irreducible recurrent chains

If \(X\) is irreducible recurrent and the payoff \(f\) has a maximum, then the optimal value is constant:

\[
v(i)=c=\max_j f(j).
\]

An optimal stopping time is the first visit to the set of states where \(f\) attains its maximum.

Reason: the chain eventually hits that set from every starting state with probability 1. Since no payoff can exceed \(c\), waiting for a maximum-payoff state achieves the best possible expected value.

## 3.3 Example 3.7: supremum may not be attained

The chapter gives an infinite-state irreducible recurrent chain on \(E=\{1,2,3,\ldots\}\) with payoff

\[
f(j)=1-\frac1j.
\]

The supremum of possible payoffs is \(1\), but no state has payoff exactly \(1\). Therefore

\[
v(i)=1
\]

for all \(i\), but no stopping time can achieve exactly \(1\). For every stopping time \(T\),

\[
E_i[f(X_T)]<1.
\]

This illustrates an important distinction:

- the value \(v\) may exist;
- the optimal stopping time may fail to exist.

## 3.4 Finite-state optimal stopping theorem

If the state space \(E\) is finite, then an optimal stopping time exists. Let

\[
A=\{j\in E: v(j)=f(j)\}.
\]

Then the first hitting time of \(A\),

\[
T_0=\inf\{n\ge0:f(X_n)=v(X_n)\},
\]

is optimal.

Thinking model:

- Continue in states where \(v(i)>f(i)\): waiting has strictly higher value.
- Stop in states where \(v(i)=f(i)\): immediate payoff already equals the best possible value.

In [ ]:
stop_set = [states[i] for i in range(len(states)) if abs(v_game[i] - payoff[i]) < 1e-10]
continue_set = [states[i] for i in range(len(states)) if abs(v_game[i] - payoff[i]) >= 1e-10]
stop_set, continue_set

## 3.5 Lemma 3.18: hitting an excessive set gives an excessive function

Let \(A\) be a fixed set and let \(T\) be the first hitting time of \(A\). If \(g\) is excessive, then

\[
h(i)=E_i[g(X_T)]
\]

is also excessive.

The proof uses a one-step decomposition. If \(i\in A\), then \(T=0\) and \(h(i)=g(i)\). If \(i\notin A\), then

\[
h(i)=\sum_j P(i,j)h(j)=Ph(i).
\]

Since \(g\) is excessive and \(h\le g\), this gives the needed excessivity. This lemma supports the proof that the first time \(v=f\) is optimal in finite state spaces.

## Summary of Chapter 7

The chapter builds a bridge from Markov chain potential theory to optimal stopping.

| Concept | Core equation | Meaning |
|---|---:|---|
| \(\alpha\)-potential | \(R^\alpha g=E[\sum\alpha^n g(X_n)]\) | expected discounted reward |
| Resolvent | \(R^\alpha=(I-\alpha P)^{-1}\) | matrix form of discounted future |
| Fixed reward equation | \(f=g+\alpha Pf\) | current reward plus future value |
| Excessive function | \(f\ge\alpha Pf\) | waiting one step is not better in expectation |
| Potential is excessive | \(R^\alpha g\ge\alpha P R^\alpha g\) | accumulated rewards dominate shifted rewards |
| Optional sampling | \(f(i)\ge E_i[\alpha^T f(X_T)]\) | stopping later does not improve excessive value |
| Optimal stopping value | \(v=\sup_T E_i[f(X_T)]\) | best achievable expected payoff |
| Main theorem | \(v=\) minimal excessive majorant of \(f\) | optimal stopping via potential theory |
| Finite-state rule | stop when \(v=f\) | first contact with payoff region is optimal |

Operationally, for finite chains:

1. start with payoff \(f\);
2. repeatedly compute \(v\leftarrow\max(f,Pv)\), or solve the corresponding linear program;
3. stop in states where \(v=f\);
4. continue where \(v>f\).